# DeepSpeech

An **end-to-end** speech-to-text engine: audio features in, characters out, trained
with **CTC** — no hand-built phoneme dictionary or HMM aligner. Popularized by Baidu's
*Deep Speech* paper (2014) and shipped as **Mozilla DeepSpeech**, an open, offline,
streaming ASR you can run on a laptop or a Raspberry Pi.

**Domain:** Speech & Audio  ·  **runnable:** yes

> Heads-up: Mozilla **archived** DeepSpeech in 2021. The architecture and CTC ideas are still
> foundational, and its direct successor is **Coqui STT**. For new work most people now reach for
> Whisper or wav2vec 2.0 — see section 7.

## 1. What & Why

**What it is.** DeepSpeech is an **end-to-end automatic speech recognition (ASR)** system. It
replaces the classic ASR pipeline (acoustic model + pronunciation lexicon + HMM + separate decoder)
with a single neural network trained directly on `(audio, transcript)` pairs. The network reads
acoustic features and emits a probability distribution over **characters** for each ~20 ms frame;
the **CTC** loss handles the fact that you never told it *which* frame maps to which letter.

**The problem it solves.** Traditional ASR needed a phoneme dictionary, forced alignment, and a
brittle stack of components tuned by specialists. DeepSpeech showed you can get competitive accuracy
by throwing a deep RNN + CTC at lots of transcribed audio and letting it learn the alignment itself.
Mozilla's implementation added the practical bits: a **streaming**, unidirectional model, an external
**KenLM language-model scorer**, TensorFlow-Lite builds, and pretrained English weights — so you get
**offline, private, on-device** transcription with no API call.

**When to reach for it.**
- You need **fully offline / on-device** STT (privacy, air-gapped, edge, Raspberry Pi).
- You want a **streaming** transcriber that emits partial results as audio arrives.
- You're studying how end-to-end CTC ASR works, or maintaining a legacy DeepSpeech/Coqui deployment.

**When not to.** For a brand-new project wanting best accuracy with the least effort, prefer
**Whisper** (multilingual, punctuation, robust to noise) or **wav2vec 2.0** (fine-tune on your
domain). DeepSpeech is archived, English-centric out of the box, and noticeably weaker on noisy or
accented speech than modern models.

## 2. Mental Model

Think **"OCR for sound, read left-to-right, one character at a time."** The network never sees words
or phonemes — it slides over the audio and, for each tiny frame, guesses a letter (or *nothing*).

```
16 kHz mono PCM
   │
   ▼  feature extraction  (MFCC: 25 ms window, 10 ms hop  ->  ~100 frames/sec)
features  x₁ x₂ … xT       ← 26 MFCC coeffs per frame
   │
   ▼  context stacking      each frame sees ±9 neighbors  -> 19×26 = 494 inputs
   ▼  3 dense (ReLU) layers
   ▼  1 recurrent layer (LSTM; unidirectional = streaming-capable)
   ▼  1 dense layer
   ▼  softmax over the alphabet + BLANK
char probs  p₁ p₂ … pT     ← one distribution per ~20 ms frame
   │
   ├─ training:   CTC loss aligns "h e l l o" to T frames automatically
   └─ inference:  CTC decode (greedy, or beam search + KenLM scorer) -> "hello"
```

Two mantras:
1. **"CTC does the alignment for you."** You give it audio and the *string* — not the timing. The
   **blank** token lets the model say "no new letter here" and lets it emit real double letters.
2. **"Acoustic model proposes, language model disposes."** The net gives raw character probabilities;
   the external **`.scorer`** (a KenLM n-gram LM) re-ranks beams so output looks like real words.

## 3. Key Concepts

- **End-to-end ASR.** One network maps audio → text. No separate pronunciation lexicon, no HMM, no
  forced alignment step. Contrast with the classic GMM-HMM / hybrid DNN-HMM pipelines.
- **MFCC features.** Mel-Frequency Cepstral Coefficients. Audio is framed (25 ms window, 10 ms hop →
  ~100 frames/s) and each frame is reduced to ~26 coefficients capturing the spectral envelope.
- **Context window.** DeepSpeech feeds each frame **plus 9 frames of past and future context**
  (2·9+1 = 19 frames) into the first layer → 19 × 26 = **494** input features per step.
- **CTC (Connectionist Temporal Classification).** The loss/decoding trick that aligns an
  unsegmented frame sequence to a shorter label string. Introduces a **blank** symbol; training sums
  over all alignments, inference collapses repeats and drops blanks.
- **Blank token.** "Emit nothing new this frame." Essential for variable-rate speech and for keeping
  genuine double letters ("ll", "ee") from collapsing.
- **Acoustic model (`.pbmm` / `.tflite`).** The trained neural net. `.pbmm` is a memory-mapped
  TensorFlow graph; `.tflite` is the smaller, quantized, edge-friendly build.
- **External scorer (`.scorer`).** A packaged **KenLM** n-gram language model + trie used during
  **beam-search** decoding to bias toward plausible word sequences. Optional but a big WER win.
- **Streaming.** v0.6+ uses a **unidirectional** LSTM so it can transcribe incrementally:
  `createStream()` → `feedAudioContent(chunk)` → `intermediateDecode()` → `finishStream()`.
- **16 kHz, 16-bit, mono.** The input contract. Audio must be `int16` PCM at 16 kHz; resample/convert
  first or accuracy collapses.
- **WER (Word Error Rate).** The standard accuracy metric: (substitutions + insertions + deletions) /
  reference words. Mozilla's v0.9 English model is ~7.x% WER on LibriSpeech clean.

## 4. Setup

Mozilla's `deepspeech` Python package ships the runtime plus a CLI. **Important:** the package
targets **Python 3.5–3.9** and is no longer maintained (the project is archived). On modern Python
(3.10+) the wheel typically won't install — that's why the real-inference cell below is **gated**.

```bash
# On Python <=3.9 only:
%pip install deepspeech            # CPU runtime (use deepspeech-gpu for CUDA)

# Pretrained English model + KenLM scorer (~190 MB + ~950 MB):
curl -LO https://github.com/mozilla/DeepSpeech/releases/download/v0.9.3/deepspeech-0.9.3-models.pbmm
curl -LO https://github.com/mozilla/DeepSpeech/releases/download/v0.9.3/deepspeech-0.9.3-models.scorer

# CLI smoke test:
deepspeech --model deepspeech-0.9.3-models.pbmm \
           --scorer deepspeech-0.9.3-models.scorer \
           --audio my_audio.wav
```

The two **Worked Examples** below need only **numpy** — they reproduce the feature/context math and
the CTC decode that govern *every* DeepSpeech model, so they always run in a fresh kernel. The third
example shows the real `deepspeech` API and is gated behind an env var + model files.

In [ ]:
import numpy as np

# The first two examples use only numpy. `deepspeech` itself (if present) is optional.
try:
    import deepspeech
    print("deepspeech:", deepspeech.version())
except Exception as e:
    print("deepspeech runtime not available (expected on Python 3.10+):",
          type(e).__name__)
    print("-> pure-numpy examples below still run and teach the core mechanics.")

print("numpy:", np.__version__)

## 5. Worked Examples

1. **Feature & context math** — how a clip of 16 kHz audio becomes ~100 MFCC frames, and why each
   step feeds the network 494 numbers. This is the shape that drives the whole pipeline.
2. **Greedy CTC decoding** — collapse a frame-level argmax into text exactly as DeepSpeech does at
   inference (and see why the blank token matters).
3. **Real inference** (gated) — the `deepspeech.Model` → `stt()` call shape, run only when the
   package and model files are present.

In [ ]:
# Example 1 — From waveform to network input: framing, hop, and context stacking.
SR        = 16_000      # required sample rate (Hz)
WIN_MS    = 25          # MFCC analysis window
HOP_MS    = 10          # stride between frames
N_MFCC    = 26          # cepstral coefficients per frame (DeepSpeech default)
N_CONTEXT = 9           # frames of past AND future context fed with each frame

win  = int(SR * WIN_MS / 1000)   # samples per window
hop  = int(SR * HOP_MS / 1000)   # samples per hop

def n_frames(num_samples):
    # Number of full analysis windows that fit (framing with no padding).
    if num_samples < win:
        return 0
    return 1 + (num_samples - win) // hop

print(f"window = {win} samples ({WIN_MS} ms), hop = {hop} samples ({HOP_MS} ms)")
print(f"frame rate = {1000/HOP_MS:.0f} frames/sec  ->  one frame every {HOP_MS} ms\n")

for secs in (1, 5, 10):
    n = secs * SR
    f = n_frames(n)
    print(f"{secs:>2}s audio = {n:>7} samples -> {f:>4} MFCC frames (~{f/secs:.0f} fps)")

# Each frame is concatenated with +/- N_CONTEXT neighbors before the first dense layer.
input_width = (2 * N_CONTEXT + 1) * N_MFCC
print(f"\nContext window = 2*{N_CONTEXT}+1 = {2*N_CONTEXT+1} frames")
print(f"Input width per step = {2*N_CONTEXT+1} x {N_MFCC} = {input_width} features")

In [ ]:
# Example 2 — Greedy CTC decoding: turn per-frame character predictions into a string.
# DeepSpeech emits, for each ~20ms frame, a distribution over the alphabet + a BLANK.
# Greedy decode = argmax per frame, then (1) collapse consecutive duplicates, (2) drop blanks.
BLANK = "_"
alphabet = [BLANK, "h", "e", "l", "o", " "]   # tiny toy alphabet (real one ~29 chars)

# Pretend these are per-frame argmax ids for "hello". Note repeats (the model fires the
# same letter over several frames) and the BLANK separating the two l's so they survive.
frame_ids    = [0, 1, 1, 0, 2, 2, 0, 3, 3, 0, 3, 0, 4, 4, 4, 0, 0]
frame_tokens = [alphabet[i] for i in frame_ids]
print("Per-frame argmax:", "".join(frame_tokens))

def ctc_greedy_decode(tokens, blank=BLANK):
    out, prev = [], None
    for t in tokens:
        if t != prev and t != blank:   # a NEW, non-blank symbol
            out.append(t)
        prev = t                       # blank also resets `prev`
    return "".join(out)

print("Decoded text   :", repr(ctc_greedy_decode(frame_tokens)))

# Without the separating blank, the two l's collapse into one:
no_blank = [alphabet[i] for i in [1, 2, 3, 3, 4]]   # h e l l o, no blank between l's
print("No-blank l l   :", repr(ctc_greedy_decode(no_blank)), "<- double letter lost")

In [ ]:
# Example 3 — Real inference with the deepspeech runtime (gated).
# Needs: pip-installed `deepspeech` (Python <=3.9) AND the model files present, plus
# RUN_DEEPSPEECH=1. Otherwise we just print the exact call shape you'd run.
import os

MODEL  = "deepspeech-0.9.3-models.pbmm"
SCORER = "deepspeech-0.9.3-models.scorer"

ready = (os.getenv("RUN_DEEPSPEECH")
         and os.path.exists(MODEL)
         and "deepspeech" in globals())

if ready:
    import wave
    from deepspeech import Model

    model = Model(MODEL)                       # load acoustic model
    if os.path.exists(SCORER):
        model.enableExternalScorer(SCORER)     # add KenLM language model

    with wave.open("my_audio.wav", "rb") as w: # MUST be 16 kHz, 16-bit, mono
        frames = w.readframes(w.getnframes())
    audio = np.frombuffer(frames, dtype=np.int16)

    text = model.stt(audio)                     # one-shot transcription
    print("transcript:", repr(text))
else:
    print("RUN_DEEPSPEECH not set / runtime or model missing — showing call shape:\n")
    print("  from deepspeech import Model")
    print("  model = Model('deepspeech-0.9.3-models.pbmm')")
    print("  model.enableExternalScorer('deepspeech-0.9.3-models.scorer')")
    print("  audio = np.frombuffer(wav_16k_mono_int16_bytes, np.int16)")
    print("  text  = model.stt(audio)")
    print("\n  # Streaming variant:")
    print("  s = model.createStream()")
    print("  s.feedAudioContent(chunk); partial = s.intermediateDecode()")
    print("  final = s.finishStream()")

## 6. Gotchas & Pitfalls

- **The project is archived.** Mozilla stopped development in 2021; no new models, fixes, or
  Python-version support. The maintained fork is **Coqui STT** (same API/format). Plan accordingly.
- **Python version wall.** The `deepspeech` wheels target **Python 3.5–3.9**. On 3.10+ `pip install`
  usually fails — use a 3.9 venv/conda env, Docker, or Coqui STT. (That's why Example 3 is gated.)
- **Audio contract is strict: 16 kHz, 16-bit, mono.** Wrong sample rate, stereo, or non-`int16`
  silently tanks accuracy or errors. Convert first (e.g. `ffmpeg -ar 16000 -ac 1 -sample_fmt s16`).
- **Pass `int16` samples, not float.** `model.stt()` expects a numpy `int16` array. Feeding float
  PCM in `[-1, 1]` gives garbage. Use `np.frombuffer(..., np.int16)`.
- **Forgetting the scorer hurts a lot.** Without `enableExternalScorer`, decoding is pure greedy/
  acoustic and WER jumps. The `.scorer` (KenLM) is what makes output read like real words.
- **Acoustic vs language model are separate files.** The `.pbmm`/`.tflite` is the net; the `.scorer`
  is the LM. You can swap or build a domain-specific scorer (`generate_scorer_package`) without
  retraining the acoustic model — a cheap, high-leverage customization.
- **English-only out of the box.** The released model is English. Other languages mean training your
  own (lots of data + GPU time) or finding a community model — a real effort.
- **Streaming ≠ free.** Use `createStream()`/`feedAudioContent()`/`intermediateDecode()` for live
  audio; calling `stt()` repeatedly on chunks loses cross-chunk context and quality.
- **`.pbmm` vs `.tflite`.** `.pbmm` is memory-mapped and faster on desktop/server; `.tflite` is
  smaller and meant for mobile/edge. Match the build to the platform.
- **Greedy demos ≠ production.** Beam width and `lm_alpha`/`lm_beta` scorer weights matter; tune them
  on a dev set instead of trusting defaults for a real deployment.

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|
| **DeepSpeech** (Mozilla) | Fully **offline / on-device**, **streaming**, small footprint, simple `stt()` API, swappable KenLM scorer | **Archived**; Python <=3.9; English-centric; weaker on noise/accents than modern models | Edge/offline/streaming, legacy maintenance, learning CTC ASR |
| **Coqui STT** | Maintained DeepSpeech successor, same format/API, more languages | Smaller community than the big new models | You like DeepSpeech's design but want upkeep |
| **Whisper** (OpenAI) | Turnkey **multilingual** with punctuation & casing, robust to noise/accents, easy to run | Heavier/slower; not truly streaming (chunked); can hallucinate on silence | Best out-of-the-box accuracy with minimal effort |
| **wav2vec 2.0 / MMS** | Open weights, **fine-tune on your domain** with little data; great feature extractor | Raw CTC output uppercase/unpunctuated; needs an LM for best WER | Custom domain + few labels, self-hosted embeddings |
| **NVIDIA NeMo** (Conformer/RNN-T) | Production-grade **streaming** ASR, strong toolkit | Heavier framework, GPU-centric | Building a full production streaming stack |
| **Cloud STT** (Google/AWS/Azure) | Highest accuracy, many languages, zero ops | Per-minute cost, network dependency, **privacy** | Online service where data leaving the box is fine |

**Rule of thumb:** *Offline/edge/streaming or maintaining old code* → DeepSpeech (or Coqui STT).
*Greenfield, want the best transcript fast* → Whisper. *Custom domain with open weights* → wav2vec 2.0.

## 8. Resources

- **Mozilla DeepSpeech docs (readthedocs)** — usage, training, API:
  https://deepspeech.readthedocs.io/en/latest/
- **GitHub repo (archived) + v0.9.3 release (pretrained model + scorer downloads)**:
  https://github.com/mozilla/DeepSpeech/releases/tag/v0.9.3
- **Paper — "Deep Speech: Scaling up end-to-end speech recognition"** (Hannun et al., Baidu, 2014):
  https://arxiv.org/abs/1412.5567
- **CTC explainer (Distill — "Sequence Modeling with CTC")** — the core trick, visualized:
  https://distill.pub/2017/ctc/
- **Coqui STT** — the maintained successor (same model format/API):
  https://github.com/coqui-ai/STT
- **Mozilla blog — "A Journey to <10% Word Error Rate"** (context on the v0.6 model):
  https://hacks.mozilla.org/2017/11/a-journey-to-10-word-error-rate/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
SR, WIN_MS, HOP_MS, N_MFCC, N_CONTEXT = 16_000, 25, 10, 26, 9


def n_frames(num_samples, sample_rate=SR):
    ...


def wer_counts(reference, hypothesis):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE